In [1]:
import ROOT

Abrindo arquivo

In [2]:
arquivo = ROOT.TFile.Open("AO2D.root")

In [3]:
diretorio = arquivo.Get("DF_2453490038269370723")
diretorio.ls()

TDirectoryFile*		DF_2453490038269370723	DF_2453490038269370723
 KEY: TTree	O2collision_001;1	Collision tree
 KEY: TTree	DbgEventExtra;1	Collision extra
 KEY: TTree	O2track;1	Barrel tracks Parameters
 KEY: TTree	O2trackcov;1	Barrel tracks Covariance
 KEY: TTree	O2trackextra_002;1	Barrel tracks Extra
 KEY: TTree	O2fwdtrack;1	Forward tracks Parameters
 KEY: TTree	O2fwdtrackcov;1	Forward tracks Covariances
 KEY: TTree	O2calo;1	Calorimeter cells
 KEY: TTree	O2calotrigger;1	Calorimeter triggers
 KEY: TTree	O2zdc_001;1	ZDC
 KEY: TTree	O2fv0a;1	FV0A
 KEY: TTree	O2fv0c;1	FV0C
 KEY: TTree	O2ft0;1	FT0
 KEY: TTree	O2fdd_001;1	FDD
 KEY: TTree	O2v0_002;1	V0s
 KEY: TTree	O2run2otfv0;1	V0s on the fly
 KEY: TTree	O2cascade_001;1	Cascades
 KEY: TTree	O2bc_001;1	BC info
 KEY: TTree	O2run2bcinfo_001;1	Run 2 BC Info
 KEY: TTree	O2origin;1	DF ids
 KEY: TTree	O2hmpid_001;1	HMPID info
 KEY: TTree	O2hf2prong;1	HF 2 prong candidates
 KEY: TTree	O2hf3prong;1	HF 3 prong candidates
 KEY: TTree	O2hfcascade;1	HF cas

In [4]:
tree = diretorio.Get("O2track")

In [ ]:
estrutura = {}

for key in arquivo.GetListOfKeys():
    if key.GetClassName() == "TDirectoryFile":
        nome_dir = key.GetName()
        diretorio = arquivo.Get(nome_dir)

        trees = []

        for subkey in diretorio.GetListOfKeys():
            obj = diretorio.Get(subkey.GetName())
            if obj.InheritsFrom("TTree"):
                trees.append(subkey.GetName())

        estrutura[nome_dir] = trees

print(f"Número de diretórios encontrados: {len(estrutura)}")
print("\nESTRUTURA")

for nome_dir, trees in list(estrutura.items())[:5]:
    print(f"\n{nome_dir}")
    for t in trees:
        print(f"   └── {t}")


Número de diretórios encontrados: 5

ESTRUTURA

DF_2453490008951090619
   └── O2collision_001
   └── DbgEventExtra
   └── O2track
   └── O2trackcov
   └── O2trackextra_002
   └── O2fwdtrack
   └── O2fwdtrackcov
   └── O2calo
   └── O2calotrigger
   └── O2zdc_001
   └── O2fv0a
   └── O2fv0c
   └── O2ft0
   └── O2fdd_001
   └── O2v0_002
   └── O2run2otfv0
   └── O2cascade_001
   └── O2bc_001
   └── O2run2bcinfo_001
   └── O2origin
   └── O2hmpid_001
   └── O2hf2prong
   └── O2hf3prong
   └── O2hfcascade
   └── O2hfdstar
   └── O2run2trackextra_001
   └── O2pmd
   └── O2pidtpcel
   └── O2pidtpcmu
   └── O2pidtpcpi
   └── O2pidtpcka
   └── O2pidtpcpr
   └── O2pidtpcde
   └── O2pidtpctr
   └── O2pidtpche
   └── O2pidtpcal
   └── O2centrun2v0m
   └── O2centrun2v0a
   └── O2centrun2cl0
   └── O2centrun2cl1
   └── O2centrun2remu5
   └── O2centrun2remu8
   └── O2fmd

DF_2453490038269370723
   └── O2collision_001
   └── DbgEventExtra
   └── O2track
   └── O2trackcov
   └── O2trackextra_002
   └─

In [6]:
primeiro_dir = list(estrutura.keys())[0]
diretorio = arquivo.Get(primeiro_dir)

tree = diretorio.Get("O2track")
#tree_cent = diretorio.Get("O2centrun2v0m")

print(f"Número de branches: {tree.GetListOfBranches().GetEntries()}")
print(f"Número de entradas (tracks): {tree.GetEntries()}")
print()

for branch in tree.GetListOfBranches():
    print(branch.GetName())


Número de branches: 9
Número de entradas (tracks): 1126821

fIndexCollisions
fTrackType
fX
fAlpha
fY
fZ
fSnp
fTgl
fSigned1Pt


In [7]:
rdf = ROOT.RDataFrame(tree)

print(f"Número de entradas no RDataFrame: {rdf.Count().GetValue()}")

Número de entradas no RDataFrame: 1126821


In [8]:
rdf_vars = (
    rdf
    .Define("pt", "1.0/fabs(fSigned1Pt)")
    .Define("eta", "asinh(fTgl)")
    .Define("phi", "fAlpha + asin(fSnp)")
)

aqui poderia-se aplicar cortes com `.Filter`, tipo 

rdf_filtrado = rdf_vars.Filter("pt > 0 && pt < 50")

In [9]:
# hist momento transversal
h_pt = rdf_vars.Histo1D(
    ("h_pt", "Distribuicao de p_{T};p_{T} (GeV/c);Contagens", 100, 0, 20),
    "pt"
)

In [10]:
# hist pseudorrapidez
h_eta = rdf_vars.Histo1D(
    ("h_eta", "Distribuicao de #eta;#eta;Contagens", 100, -1.5, 1.5),
    "eta"
)

In [11]:
# hist angulo azimutal
h_phi = rdf_vars.Histo1D(
    ("h_phi", "Distribuicao de #phi;#phi (rad);Contagens", 100, -3.5, 3.5),
    "phi"
)

In [12]:
# hist posicoes
h_fx = rdf_vars.Histo1D(
    ("h_fx", "Distribuicao de fX;fX (cm);Contagens", 100, -5, 5),
    "fX"
)

h_fy = rdf_vars.Histo1D(
    ("h_fy", "Distribuicao de fY;fY (cm);Contagens", 100, -5, 5),
    "fY"
)

h_fz = rdf_vars.Histo1D(
    ("h_fz", "Distribuicao de fZ;fZ (cm);Contagens", 100, -20, 20),
    "fZ"
)

In [13]:
c1 = ROOT.TCanvas("c2", "pT (log)", 800, 600)
c1.SetLogy()
h_pt.Draw()
c1.SaveAs("pt_distribuicao_log.png")

Info in <TCanvas::Print>: png file pt_distribuicao_log.png has been created


In [14]:
c3 = ROOT.TCanvas("c3", "eta", 800, 600)
h_eta.Draw()
c3.SaveAs("eta_distribuicao.png")

c4 = ROOT.TCanvas("c4", "phi", 800, 600)
h_phi.Draw()
c4.SaveAs("phi_distribuicao.png")

Info in <TCanvas::Print>: png file eta_distribuicao.png has been created
Info in <TCanvas::Print>: png file phi_distribuicao.png has been created


In [15]:
c5 = ROOT.TCanvas("c5", "posicoes", 1200, 400)
c5.Divide(3, 1)

c5.cd(1)
h_fx.Draw()

c5.cd(2)
h_fy.Draw()

c5.cd(3)
h_fz.Draw()

c5.SaveAs("posicoes_distribuicao.png")

Info in <TCanvas::Print>: png file posicoes_distribuicao.png has been created


In [ ]:
h_pt_hist = h_pt.GetValue()

print(f"Entradas: {h_pt_hist.GetEntries():.0f}")
print(f"Media (p_T): {h_pt_hist.GetMean():.3f} GeV/c")
print(f"RMS: {h_pt_hist.GetRMS():.3f} GeV/c")
print(f"Valor maximo: {h_pt_hist.GetMaximum():.0f}")

Entradas: 1126821
Media (p_T): 0.516 GeV/c
RMS: 0.717 GeV/c
Valor maximo: 206763 (bin 1)
